# Financeiro: previsto, informado e confirmado

Demonstração local com corpus fictício e fixture de extração. Não usa API nem dados da JMS. O projeto operacional é criado a partir de uma interpretação aceita; orçamento e pagamento exigem ações explícitas separadas.


In [ ]:
import sys, tempfile
from pathlib import Path
raiz = Path.cwd()
if raiz.name == 'notebooks': raiz = raiz.parent
sys.path.insert(0,str(raiz/'src'))
from secretaria.banco import Banco, Contexto
from secretaria.extracao import analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
from secretaria.projetos_fontes import ProjetosFontes
from secretaria.financeiro import Financeiro


In [ ]:
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta)/'demo.sqlite3') as banco:
        banco.preparar_demo()
        ana = Contexto('demo','ana')
        corpus = banco.selecionar_corpus(ana,'Demonstração fictícia')
        banco.importar(ana,corpus,raiz/'dados/ficticios/conversa_empresa.txt')
        analise = analisar_snapshot(banco.snapshot_analise(ana,corpus),criar_modelo_fixture(),'fixture_manual')
        fila = FilaRevisao(banco)
        proposta = fila.registrar_resultado(ana,analise)['ids'][0]
        fila.decidir(ana,proposta,1,'aceitar','Fonte conferida para esta demonstração.')
        projetos = ProjetosFontes(banco)
        previa = projetos.prever(ana,proposta)
        projeto_id = projetos.confirmar(ana,proposta,previa['versao'])['projeto_id']
        financeiro = Financeiro(banco)
        orcamento_id = financeiro.criar_orcamento(ana,projeto_id,1_000_000,30)['orcamento_id']
        assert financeiro.relatorio(ana,projeto_id)['projetos'][0]['saldo_centavos'] is None
        financeiro.aceitar_orcamento(ana,orcamento_id)
        entrada, restante = financeiro.definir_parcelas(ana,orcamento_id,[
            {'descricao':'Entrada','valor_centavos':300_000},
            {'descricao':'Restante','valor_centavos':700_000}])['parcelas_ids']
        pagamento_id = financeiro.informar_pagamento(ana,entrada,100_000,'2026-11-11','fixture-1')['pagamento_id']
        antes = financeiro.relatorio(ana,projeto_id)['projetos'][0]
        financeiro.confirmar_pagamento(ana,pagamento_id)
        depois = financeiro.relatorio(ana,projeto_id)['projetos'][0]
        print('Antes da confirmação:',antes['saldo_centavos'],'centavos')
        print('Depois da confirmação:',depois['saldo_centavos'],'centavos')
        assert antes['saldo_centavos'] == 1_000_000
        assert depois['saldo_centavos'] == 900_000


O percentual acordado de 30% não comprova pagamento. O relato de recebimento no chat também não entra no saldo. A confirmação operacional é registrada em tabela própria com autoria e auditoria.
